# DistilBERT Fine-Tuning — Notebook 1: Training

**Run this notebook once on a GPU (e.g. Google Colab T4).**

It produces two outputs in `./distilbert_finetuned/`:
1. The trained model weights + tokenizer
2. `test_texts.npy` and `test_labels.npy` — the exact test split,
   saved so Notebook 2 evaluates on the same rows without re-splitting.

**You do not need to re-run this notebook** unless you want to retrain from scratch.
All evaluation and inference is handled in Notebook 2.

## 1. Imports & Configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# Uncomment if running on a fresh machine (not needed on Colab)
# !pip install transformers scikit-learn torch tqdm

import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Training will be very slow on CPU.")

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_NAME   = 'distilbert-base-uncased'
MAX_LEN      = 256
BATCH_SIZE   = 16
NUM_EPOCHS   = 3
LR           = 2e-5
WARMUP_RATIO = 0.1
DROPOUT      = 0.3
SAMPLE_SIZE  = None   # set to e.g. 20_000 to train on a subset; None = full dataset
SAVE_DIR     = '/content/drive/MyDrive/Colab Notebooks/Thesis Notebooks and Data/distilbert_finetuned'

print(f"\nConfig: MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}, LR={LR}")

Device: cuda
GPU: Tesla T4

Config: MAX_LEN=256, BATCH_SIZE=16, EPOCHS=3, LR=2e-05


## 2. Load & Prepare Data

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/WELFake_Dataset.csv')
df = df.iloc[:, 1:]                          # drop ID column
df = df.dropna(subset=['text'])
df['title']   = df['title'].fillna('')
df['content'] = df['title'] + ' ' + df['text']

print(f"Dataset: {len(df):,} articles")
print(f"  Real (0): {(df['label']==0).sum():,}") # flip from whats given on WELFAKE: 0=real, 1=fake
print(f"  Fake (1): {(df['label']==1).sum():,}")

if SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"\nUsing subsample of {SAMPLE_SIZE:,} articles.")

Dataset: 72,095 articles
  Real (0): 35,028
  Fake (1): 37,067


In [4]:
# ── Train / Validation / Test split ──────────────────────────────────────────
# IMPORTANT: the test split is saved to disk at the end of this notebook
# so Notebook 2 evaluates on exactly these rows.

X = df['content'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f"Train:      {len(X_train):,} ({y_train.mean():.1%} fake)")
print(f"Validation: {len(X_val):,}  ({y_val.mean():.1%} fake)")
print(f"Test:       {len(X_test):,}  ({y_test.mean():.1%} fake)")

Train:      50,466 (51.4% fake)
Validation: 10,814  (51.4% fake)
Test:       10,815  (51.4% fake)


## 3. Tokenisation & Dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: vocab size = {tokenizer.vocab_size:,}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: vocab size = 30,522


In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = FakeNewsDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = FakeNewsDataset(X_val,   y_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")

batch = next(iter(train_loader))
print(f"\nBatch shapes — input_ids: {batch['input_ids'].shape}, "
      f"attention_mask: {batch['attention_mask'].shape}, "
      f"label: {batch['label'].shape}")

Train batches: 3,155
Val batches:   676

Batch shapes — input_ids: torch.Size([16, 256]), attention_mask: torch.Size([16, 256]), label: torch.Size([16])


## 4. Model Setup

In [ ]:
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    seq_classif_dropout=DROPOUT,
)
distilbert_model = distilbert_model.to(device)

total_params     = sum(p.numel() for p in distilbert_model.parameters())
trainable_params = sum(p.numel() for p in distilbert_model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     66,955,010
Trainable parameters: 66,955,010


In [ ]:
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in distilbert_model.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n, p in distilbert_model.named_parameters()
                if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = AdamW(optimizer_grouped_parameters, lr=LR)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps:         {warmup_steps:,}")

Total training steps: 9,465
Warmup steps:         946


## 5. Training Loop

In [ ]:
def evaluate(model, loader, desc='Eval'):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0.0
best_epoch  = 0
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 60)
print("FINE-TUNING DISTILBERT")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    distilbert_model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")

    for step, batch in enumerate(pbar):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = distilbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(distilbert_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{running_loss/(step+1):.4f}",
                          'lr':   f"{scheduler.get_last_lr()[0]:.2e}"})

    avg_train_loss = running_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────────────────
    val_loss, val_acc, val_f1 = evaluate(
        distilbert_model, val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]"
    )

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print(f"  Train loss: {avg_train_loss:.4f}")
    print(f"  Val   loss: {val_loss:.4f}  |  acc: {val_acc:.4f}  |  F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        distilbert_model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✅ New best — model saved to {SAVE_DIR}/")

print(f"\nTraining complete. Best val F1 = {best_val_f1:.4f} at epoch {best_epoch}.")

FINE-TUNING DISTILBERT


Epoch 1/3 [Train]:   0%|          | 0/3155 [00:00<?, ?it/s]

Epoch 1/3 [Val]:   0%|          | 0/676 [00:00<?, ?it/s]


Epoch 1/3
  Train loss: 0.0988
  Val   loss: 0.0305  |  acc: 0.9925  |  F1: 0.9925


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 3/3 [Train]:   0%|          | 0/3155 [00:00<?, ?it/s]

Epoch 3/3 [Val]:   0%|          | 0/676 [00:00<?, ?it/s]


Epoch 3/3
  Train loss: 0.0035
  Val   loss: 0.0322  |  acc: 0.9943  |  F1: 0.9943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ New best — model saved to ./distilbert_finetuned/

Training complete. Best val F1 = 0.9943 at epoch 3.


## 6. Save Training History & Test Split

This is the critical step that lets Notebook 2 work independently.
We save:
- The training history (loss/accuracy curves)
- The **exact test rows** so Notebook 2 evaluates on the same data

In [ ]:
# Save test split so Notebook 2 uses exactly the same rows
np.save(os.path.join(SAVE_DIR, 'test_texts.npy'),  X_test)
np.save(os.path.join(SAVE_DIR, 'test_labels.npy'), y_test)

# Save training history for plotting in Notebook 2
np.save(os.path.join(SAVE_DIR, 'training_history.npy'), history)

print(f"Saved to {SAVE_DIR}/")
print(f"  test_texts.npy   — {len(X_test):,} articles")
print(f"  test_labels.npy  — {len(y_test):,} labels")
print(f"  training_history.npy")
print(f"\nAll files in {SAVE_DIR}/:")
print(os.listdir(SAVE_DIR))

Saved to ./distilbert_finetuned/
  test_texts.npy   — 10,815 articles
  test_labels.npy  — 10,815 labels
  training_history.npy

All files in ./distilbert_finetuned/:
['model.safetensors', 'training_history.npy', 'test_labels.npy', 'test_texts.npy', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


## 7. Download Everything

Run this to download the full `distilbert_finetuned/` folder as a zip.
Keep this zip — it contains everything Notebook 2 needs.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('distilbert_finetuned', 'zip', '.', 'distilbert_finetuned')
files.download('distilbert_finetuned.zip')
print("Download triggered.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


In [5]:
print("Sample label=0 (should be REAL journalism):")
for text in X_test[y_test == 0][:2]:
    print(f"  {text[:100]}")
print("\nSample label=1 (should be FAKE clickbait):")
for text in X_test[y_test == 1][:2]:
    print(f"  {text[:100]}")

Sample label=0 (should be REAL journalism):
  Nearly 400 die as Myanmar army steps up crackdown on Rohingya militants COX S BAZAR, Bangladesh (Reu
  Poland ready to defend migration stance in EU top court WARSAW (Reuters) - Poland is ready to defend

Sample label=1 (should be FAKE clickbait):
  FLASHBACK - Hillary Clinton’s ‘KKK’ Smear Against Trump was Democrat Strategy November 21, 2016 By 2
   Seth MacFarlane Wants His Fellow Bernie Supporters To Know Something VERY Important Very outspoken 


In [12]:
np.save(os.path.join(SAVE_DIR, 'test_texts.npy'),  X_test)
np.save(os.path.join(SAVE_DIR, 'test_labels.npy'), y_test)
